In [5]:

# ------------------------
# Parameters block (Papermill will automatically replace these)
# ------------------------
output_bucket_path = "gs://langchain_bucket_arseny/rag/embeddings/results/"
timestamp = "20250709"
project_id = "grand-voltage-465301-e8"

# ------------------------
# BigQuery configuration
# ------------------------
from google.cloud import bigquery

bq_client = bigquery.Client(project=project_id)

# Dataset and table name
dataset_id = "dataset_for_rag_embeddings"
table_id = "transformers_embeddings"

table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ------------------------
# Query 1: Count number of chunks per document
# ------------------------
query1 = f"""
SELECT
    source_file,
    COUNT(*) AS chunk_count
FROM
    `{table_ref}`
GROUP BY
    source_file
ORDER BY
    chunk_count DESC
"""

df_chunks = bq_client.query(query1).to_dataframe()
print("Chunk count query completed")
print(df_chunks.head())

# ------------------------
# Query 2: Average chunk content length
# ------------------------
query2 = f"""
SELECT
    source_file,
    AVG(LENGTH(content)) AS avg_chunk_length
FROM
    `{table_ref}`
GROUP BY
    source_file
ORDER BY
    avg_chunk_length DESC
"""

df_avg_length = bq_client.query(query2).to_dataframe()
print("Average chunk length query completed")
print(df_avg_length.head())

# ------------------------
# Query 3: Chunks containing the keyword 'sklearn'
# ------------------------
query3 = f"""
SELECT
    source_file,
    chunk_id,
    content
FROM
    `{table_ref}`
WHERE
    LOWER(content) LIKE '%sklearn%'
LIMIT 10
"""

df_keyword = bq_client.query(query3).to_dataframe()
print("Keyword query completed")
print(df_keyword.head())

# ------------------------
# Save CSV files (with timestamp)
# ------------------------
csv1 = f"transformers_chunk_count_{timestamp}.csv"
csv2 = f"transformers_avg_chunk_length_{timestamp}.csv"
csv3 = f"transformers_keyword_snippets_{timestamp}.csv"

df_chunks.to_csv(csv1, index=False)
df_avg_length.to_csv(csv2, index=False)
df_keyword.to_csv(csv3, index=False)

print("CSV files saved locally")

# ------------------------
# Upload to GCS
# ------------------------
from google.cloud import storage

storage_client = storage.Client(project=project_id)
bucket_name = output_bucket_path.split("/")[2]
bucket = storage_client.bucket(bucket_name)

def upload_file(local_path, gcs_blob_path):
    blob = bucket.blob(gcs_blob_path)
    blob.upload_from_filename(local_path)
    print(f"Uploaded to GCS: {gcs_blob_path}")

upload_file(csv1, f"results/transformers/sklearn_chunk_count.csv")
upload_file(csv1, f"results_history/transformers/{csv1}")

upload_file(csv2, f"results/transformers/sklearn_avg_chunk_length.csv")
upload_file(csv2, f"results_history/transformers/{csv2}")

upload_file(csv3, f"results/transformers/sklearn_keyword_snippets.csv")
upload_file(csv3, f"results_history/transformers/{csv3}")

print("All results uploaded to GCS")


Chunk count query completed
                                   source_file  chunk_count
0  docs/source/en/llm_tutorial_optimization.md           68
1                  docs/source/en/deepspeed.md           64
2     docs/source/en/tasks/object_detection.md           64
3                    docs/source/en/testing.md           58
4            docs/source/en/model_doc/tapas.md           56
Average chunk length query completed
                                source_file  avg_chunk_length
0                  docs/source/en/agents.md        957.000000
1                   docs/source/en/tools.md        956.000000
2         docs/source/en/model_doc/zamba.md        953.500000
3  docs/source/en/model_doc/timm_wrapper.md        950.000000
4       docs/source/en/model_doc/colpali.md        945.571429
Keyword query completed
                            source_file  \
0  docs/source/en/model_doc/imagegpt.md   

                                 chunk_id  \
0  docs/source/en/model_doc/imagegpt.md_5   

 